In [1]:
import matplotlib
# Try 'qt5' if you have PyQt installed, otherwise 'tk' is usually built-in
%matplotlib tk

import matplotlib.pyplot as plt

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d
import matplotlib.colors as mcolors

# --- Helper Class for 3D Arrows ---
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.get_proj())
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return np.min(zs)

def generate_clumped_mock_data(samples=100):
    """
    Generates 3 clumps of data close to each other on one side of a sphere.
    """
    # Define 3 centers that are close to each other (e.g., near [0.7, 0.7, 0.1])
    centers = [
        np.array([0.9, 0.1, -0.2]),
        np.array([0.4, -0.5, -0.4]),
        np.array([0.4, -0.2, 0.3])
    ]
    pitch_labels = ["Low", "Medium", "High"]

    all_embeddings = []
    stats = []

    for i, center in enumerate(centers):
        # Add noise, then immediately normalize so they sit ON the surface
        clump = center + np.random.normal(0, 0.09, size=(samples, 3))
        clump /= np.linalg.norm(clump, axis=1, keepdims=True)

        for point in clump:
            all_embeddings.append(torch.tensor(point))
            # Assign pitch based on which clump it belongs to
            stats.append({"pitch": i * 5.0 + np.random.uniform(0, 2)})

    return all_embeddings, stats

def visualize_clumped_sphere(all_spk_embs, stats):
    # 1. Prepare data
    data = torch.stack([t.detach().cpu().squeeze() for t in all_spk_embs]).numpy()
    pitches = np.array([s["pitch"] for s in stats])

    # 2. Normalize to unit sphere surface immediately
    # In a real model, embeddings are often already high-dimensional unit vectors
    projected_coords = data / np.linalg.norm(data, axis=1, keepdims=True)

    # 3. Create wireframe sphere
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x_sphere = np.outer(np.cos(u), np.sin(v))
    y_sphere = np.outer(np.sin(u), np.sin(v))
    z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))

    # 4. Plot
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    ax.computed_zorder = False

    # Replace the wireframe block with this:
    ax.plot_surface(
        x_sphere, y_sphere, z_sphere,
        color="white",       # Solid base color
        alpha=1,           # Lower alpha = more transparent; 0.8-1.0 = opaque
        shade=False,          # Adds a 3D light effect
        linewidth=0.2,
        edgecolors='black',   # This recreates the grid lines on the "front" only
        antialiased=True,
        rcount=20,
        ccount=20,
        zorder=1             # Keeps it as the base layer
    )
    ax.set_box_aspect([1,1,1])  # Required for a perfect sphere shape

    # Plot points
    point_nudge = 1.02
    scatter = ax.scatter(
        projected_coords[:, 0] * point_nudge,
        projected_coords[:, 1] * point_nudge,
        projected_coords[:, 2] * point_nudge,
        c=pitches, cmap='viridis', s=30, alpha=1.0, edgecolors='black', linewidth=0.2,
        zorder=10 # Ensure points are drawn over the surface
    )


    # --- Centroid and Arc Logic ---
    # Calculate high-dimensional centroids
    low_idx = [i for i, s in enumerate(stats) if s["pitch"] < 3]   # Group 1
    high_idx = [i for i, s in enumerate(stats) if s["pitch"] > 10] # Group 3

    c_low = data[low_idx].mean(axis=0)
    c_high = data[high_idx].mean(axis=0)

    # Project centroids to the sphere surface
    c_low /= np.linalg.norm(c_low)
    c_high /= np.linalg.norm(c_high)

    # Plot the centroids as large distinct markers
    ax.scatter(*c_low, color='pink', s=200, marker='o', edgecolors='black', label='Low Centroid', zorder=19)
    ax.scatter(*c_high, color='pink', s=200, marker='o', edgecolors='black', label='High Centroid', zorder=21)

    # Generate the Red Curve (Slerp / Great Circle Arc)
    def slerp_path(p0, p1, n_steps=50):
        omega = np.arccos(np.clip(np.dot(p0, p1), -1, 1))
        t = np.linspace(0, 1, n_steps)
        return (np.sin((1-t)[:, None] * omega) / np.sin(omega)) * p0 + \
               (np.sin(t[:, None] * omega) / np.sin(omega)) * p1

    arc = slerp_path(c_low, c_high)

    # Nudge the arc slightly off the surface so it definitely sits 'above' the wireframe
    nudge = 1.01
    ax.plot(arc[:, 0]*nudge, arc[:, 1]*nudge, arc[:, 2]*nudge,
            color='red', linewidth=5, alpha=0.9, zorder=20)

    a = Arrow3D([arc[-2, 0]*nudge, arc[-1, 0]*nudge],
                [arc[-2, 1]*nudge, arc[-1, 1]*nudge],
                [arc[-2, 2]*nudge, arc[-1, 2]*nudge],
                mutation_scale=50, lw=1, arrowstyle="-|>", color="red", zorder=22)
    ax.add_artist(a)


    # --- New Arc Logic (Medium to High) ---
    # Identify Medium group
    med_idx = [i for i, s in enumerate(stats) if 5 <= s["pitch"] <= 10]
    c_med = data[med_idx[7]] #.mean(axis=0)
    c_med /= np.linalg.norm(c_med) # Project to surface

    # Generate the dashed arc path
    arc_med_high = slerp_path(c_med, c_high)

    # Plot the dashed arc (50% of the way or full path with styling)
    # To make it go "towards" high but only 50% of the way:
    half_way_point = len(arc_med_high) // 2
    arc_segment = arc_med_high[:half_way_point]

    # Plot with dashed lines
    nudge_dashed = 1.02
    ax.plot(arc_segment[:, 0]*nudge_dashed,
            arc_segment[:, 1]*nudge_dashed,
            arc_segment[:, 2]*nudge_dashed,
            color='orange', linewidth=3, linestyle='--', alpha=0.8, zorder=25)

    # Adding the Arrow at the end of the Orange Segment
    a2 = Arrow3D([arc_segment[-2, 0]*nudge_dashed, arc_segment[-1, 0]*nudge_dashed],
                 [arc_segment[-2, 1]*nudge_dashed, arc_segment[-1, 1]*nudge_dashed],
                 [arc_segment[-2, 2]*nudge_dashed, arc_segment[-1, 2]*nudge_dashed],
                 mutation_scale=40, lw=1, arrowstyle="-|>", color="orange", zorder=30)
    ax.add_artist(a2)

    # Generate the dashed arc path
    arc_med_low = slerp_path(c_med, c_low)

    # Plot the dashed arc (50% of the way or full path with styling)
    # To make it go "towards" high but only 50% of the way:
    half_way_point = len(arc_med_low) // 2
    arc_segment = arc_med_low[:half_way_point]

    # Plot with dashed lines
    nudge_dashed = 1.02
    ax.plot(arc_segment[:, 0]*nudge_dashed,
            arc_segment[:, 1]*nudge_dashed,
            arc_segment[:, 2]*nudge_dashed,
            color='blue', linewidth=3, linestyle='--', alpha=0.8, zorder=25)

    # Adding the Arrow at the end of the Orange Segment
    a3 = Arrow3D([arc_segment[-2, 0]*nudge_dashed, arc_segment[-1, 0]*nudge_dashed],
                 [arc_segment[-2, 1]*nudge_dashed, arc_segment[-1, 1]*nudge_dashed],
                 [arc_segment[-2, 2]*nudge_dashed, arc_segment[-1, 2]*nudge_dashed],
                 mutation_scale=40, lw=1, arrowstyle="-|>", color="blue", zorder=30)
    ax.add_artist(a3)


    ######################################################################################################

    norm = mcolors.Normalize(vmin=pitches.min(), vmax=pitches.max())
    cmap = plt.get_cmap('viridis')

    # 2. Get the specific colors for low and high pitch
    # Replace 1.5 and 13.5 with the actual mean pitch of your groups if preferred
    color_low = cmap(norm(1.5))
    color_high = cmap(norm(10))

    # --- Labels ---
    label_nudge = 1.15 # Push text slightly further out than the points/lines
    # Define a consistent box style
    bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.5, alpha=0.8)

    # 1. Pink dots: "Low/high centroid"
    ax.text(c_low[0]*label_nudge, c_low[1]*label_nudge, c_low[2]*label_nudge,
            "Low \ncentroid", color=color_low, fontweight='bold', bbox=bbox_props, zorder=40)
    ax.text(c_high[0]*label_nudge, c_high[1]*label_nudge, c_high[2]*label_nudge,
            "High \ncentroid", color=color_high, fontweight='bold', bbox=bbox_props, zorder=40)

    # 2. Red Solid Arrow: "Direction of pitch"
    # Placed at the midpoint of the red arc
    mid_arc = arc[len(arc)//2]
    ax.text(mid_arc[0]*label_nudge, mid_arc[1]*label_nudge, mid_arc[2]*label_nudge,
            "Direction of pitch", color='red', fontweight='bold', bbox=bbox_props, zorder=40)

    # 3. Orange/Red Dashed Arrow: "Towards high pitch speakers"
    # Using the 'a2' arrow tip position (end of arc_segment)
    tip_high = arc_med_high[half_way_point-1]
    ax.text(tip_high[0]*label_nudge, tip_high[1]*label_nudge, tip_high[2]*label_nudge,
            "Towards high \npitch speakers", color='orange', fontweight='bold', bbox=bbox_props, zorder=40)

    # 4. Blue Dashed Arrow: "Towards low pitch speakers"
    # Using the 'a3' arrow tip position
    tip_low = arc_med_low[half_way_point-1]
    ax.text(tip_low[0]*label_nudge, tip_low[1]*label_nudge, tip_low[2]*label_nudge,
            "Towards low \npitch speakers", color='blue', fontweight='bold', bbox=bbox_props, zorder=40)
    ######################################################################################################

    # Formatting
    #ax.set_title("3 Clumps on the Same Side of the Hypersphere", fontsize=15)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.set_xlabel(""); ax.set_ylabel(""); ax.set_zlabel("")
    ax.grid(False)
    ax.set_axis_off()

    # Set the view so we can see the "clumped" side clearly
    ax.view_init(elev=-12, azim=-34, roll=-1)
    ax.set_proj_type('persp', focal_length=0.20)
    margin = 0.75
    ax.set_xlim(-margin, margin)
    ax.set_ylim(-margin, margin)
    ax.set_zlim(-margin, margin)

    plt.colorbar(scatter, ax=ax, shrink=0.5, label="Pitch")
    plt.show()

# Execution
mock_embs, mock_stats = generate_clumped_mock_data()
visualize_clumped_sphere(mock_embs, mock_stats)